# NexusAI Core-01 — Kaggle GPU Diagnostic (Synthetic Data Only)

**Purpose**: verify GPU + PyTorch CUDA + AMP + Core-01 forward/backward + checkpoint save/load.
**This notebook does NOT train on real data and does NOT start a training run.**
Run cell-by-cell. Stop if any cell errors.

In [ ]:
# Cell 1: Environment check
import sys, os, torch, platform
print("python:", sys.version)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "n/a")

In [ ]:
# Cell 2: GPU identity + memory
assert torch.cuda.is_available(), "No GPU! Set Accelerator=GPU T4 x2 in notebook settings."
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
!nvidia-smi

In [ ]:
# Cell 3: Locate project files (adjust path to your Kaggle input name)
import glob
candidates = glob.glob("/kaggle/input/**/core01", recursive=True)
print("core01 found at:", candidates)
AI_CORE = os.path.dirname(os.path.dirname(candidates[0])) if candidates else None
print("ai-core root:", AI_CORE)
sys.path.insert(0, AI_CORE)
sys.path.insert(0, os.path.join(AI_CORE, "models"))
from core01.transformer import NexusAICore01
from nexusai_core_01.config.model_config import NexusAICore01Config
print("imports OK")

In [ ]:
# Cell 4: Model init on GPU + param count + memory
cfg = NexusAICore01Config(device="cuda", precision="fp16",
                      batch_size=8, gradient_accumulation_steps=2)
model = NexusAICore01(cfg).cuda()
print("params:", model.count_parameters())
assert model.count_parameters() == 8_842_560
print("VRAM used MB:", round(torch.cuda.memory_allocated()/1e6, 1))

In [ ]:
# Cell 5: Synthetic forward + backward + optimizer step (FP32 first)
from core01.training import PretrainingEngine
import tempfile

torch.manual_seed(0)
x = torch.randint(0, 8000, (8, 512), device="cuda")
y = torch.randint(0, 8000, (8, 512), device="cuda")
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss = model(x, y)
print("FP32 loss:", loss.item())
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
opt.step(); opt.zero_grad()
print("forward+backward+step OK, VRAM MB:", round(torch.cuda.memory_allocated()/1e6,1))

In [ ]:
# Cell 6: AMP FP16 forward+backward via GradScaler
scaler = torch.amp.GradScaler("cuda")
opt.zero_grad()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss_amp = model(x, y)
print("FP16 autocast loss:", loss_amp.item(), "dtype:", loss_amp.dtype)
scaler.scale(loss_amp).backward()
scaler.unscale_(opt)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(opt); scaler.update(); opt.zero_grad()
print("AMP step OK, scaler scale:", scaler.get_scale())

In [ ]:
# Cell 7: BF16 check (T4 expected NOT supported -> code should fall back)
from core01.training import resolve_amp_dtype
print("bf16 supported:", torch.cuda.is_bf16_supported())
print("resolve bf16 ->", resolve_amp_dtype("bf16", "cuda"))
print("resolve fp16 ->", resolve_amp_dtype("fp16", "cuda"))

In [ ]:
# Cell 8: Checkpoint save + reload (to /kaggle/working = persistent)
ckdir = "/kaggle/working/ckpt_diagnostic"
engine = PretrainingEngine(model, cfg, ckdir,
                           tokenizer_path="<set-real-path>")
engine.current_step = 10
path = engine.save_checkpoint(epoch=0, train_loss=9.9, val_loss=9.9,
                              filename="diag.pt")
print("saved:", path, "size MB:", round(os.path.getsize(path)/1e6,1))

model2 = NexusAICore01(cfg).cuda()
engine2 = PretrainingEngine(model2, cfg, ckdir)
engine2.load_checkpoint("diag.pt")
same = all(torch.equal(a.cpu(), b.cpu()) for a,b in
           zip(model.state_dict().values(), model2.state_dict().values())
           if torch.is_tensor(a))
print("weights restored identical:", same, "| step:", engine2.current_step)

In [ ]:
# Cell 9: Throughput micro-benchmark (10 steps only, ~seconds)
import time
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
scaler = torch.amp.GradScaler("cuda")
torch.cuda.synchronize(); t0=time.time()
for i in range(10):
    opt.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        l = model(x, y)
    scaler.scale(l).backward()
    scaler.step(opt); scaler.update()
torch.cuda.synchronize(); dt=time.time()-t0
tok_per_s = 10*8*512/dt
print(f"10 steps in {dt:.1f}s -> {tok_per_s:,.0f} tokens/sec (FP16, batch8 seq512)")
print("VRAM peak MB:", round(torch.cuda.max_memory_allocated()/1e6,1))
print("NOTE: this is a diagnostic only. Real training needs owner approval.")